# 03a_r0_group — R0-집단 학습 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 4.1절. 질문의 3단계 라벨(풀 확률 ≥ .8인 가장 싼 집단)을 분류한다. 섞은 데이터(원천 4개, 같은 비중). RT-05와 같은 학습 함수(`train_objective`)·설정(`routing_objective_continual_v1.json`)·seed 42. test는 읽지 않는다.

**런타임: 2026.07(Python 3.12), T4 GPU.** 약 1.5~2시간. 03a·03b·03c는 서로 독립이라 다른 GPU 세션에서 동시에 돌려도 된다. 중간 저장이 없으므로 연결이 끊기면 처음부터 다시 실행한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-ours.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/router_final/requirements-train-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(REPO / 'ml')], check=True)

설치가 끝나면 **런타임 → 세션 다시 시작**을 한 번 한 뒤 아래 셀부터 실행합니다.

In [ ]:
from pathlib import Path
import os, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'
print(torch.cuda.get_device_name(0), torch.__version__)
ENV = dict(os.environ, PYTHONPATH=str(REPO / 'ml/src'))
OURS = REPO / 'ml/notebooks/router_final/ours.py'
DATA = RUN / 'ours' / 'data'
LOCAL = Path('/content/ours-r0_group')
FINAL = RUN / 'ours' / 'models' / 'r0_group'
assert not FINAL.exists(), '이미 결과가 있다. 다시 돌리려면 폴더를 다른 이름으로 옮긴 뒤 실행'

In [ ]:
# 1) 2 step 시험 학습 (몇 분)
subprocess.run([sys.executable, str(OURS), 'train', '--root', str(REPO), '--data-dir', str(DATA), '--run', 'r0_group', '--output-dir', '/content/ours-smoke', '--smoke', '--device', 'cuda'], env=ENV, check=True)

In [ ]:
# 2) 본 학습 → 끝나면 Drive로 복사
import shutil
subprocess.run([sys.executable, str(OURS), 'train', '--root', str(REPO), '--data-dir', str(DATA), '--run', 'r0_group', '--output-dir', str(LOCAL), '--device', 'cuda'], env=ENV, check=True)
FINAL.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL, FINAL)
print(sorted(p.name for p in FINAL.iterdir()))